In [20]:
import httpx
import numpy as np
import torch

In [21]:
response = httpx.get("https://pokeapi.co/api/v2/pokemon/?limit=2000")
response.raise_for_status()
data = response.json()
data

{'count': 1350,
 'next': None,
 'previous': None,
 'results': [{'name': 'bulbasaur',
   'url': 'https://pokeapi.co/api/v2/pokemon/1/'},
  {'name': 'ivysaur', 'url': 'https://pokeapi.co/api/v2/pokemon/2/'},
  {'name': 'venusaur', 'url': 'https://pokeapi.co/api/v2/pokemon/3/'},
  {'name': 'charmander', 'url': 'https://pokeapi.co/api/v2/pokemon/4/'},
  {'name': 'charmeleon', 'url': 'https://pokeapi.co/api/v2/pokemon/5/'},
  {'name': 'charizard', 'url': 'https://pokeapi.co/api/v2/pokemon/6/'},
  {'name': 'squirtle', 'url': 'https://pokeapi.co/api/v2/pokemon/7/'},
  {'name': 'wartortle', 'url': 'https://pokeapi.co/api/v2/pokemon/8/'},
  {'name': 'blastoise', 'url': 'https://pokeapi.co/api/v2/pokemon/9/'},
  {'name': 'caterpie', 'url': 'https://pokeapi.co/api/v2/pokemon/10/'},
  {'name': 'metapod', 'url': 'https://pokeapi.co/api/v2/pokemon/11/'},
  {'name': 'butterfree', 'url': 'https://pokeapi.co/api/v2/pokemon/12/'},
  {'name': 'weedle', 'url': 'https://pokeapi.co/api/v2/pokemon/13/'},
  {

In [22]:
names = [p['name'] for p in data['results']]
names[:10]

['bulbasaur',
 'ivysaur',
 'venusaur',
 'charmander',
 'charmeleon',
 'charizard',
 'squirtle',
 'wartortle',
 'blastoise',
 'caterpie']

In [23]:
alphabet = ['<S>', '<E>', '<P>'] + sorted(set().union(*names))
alphabet

['<S>',
 '<E>',
 '<P>',
 '-',
 '0',
 '1',
 '2',
 '5',
 'a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z']

In [24]:
voc = dict(enumerate(alphabet))
reversed_voc = {v: k for k, v in voc.items()}
voc

{0: '<S>',
 1: '<E>',
 2: '<P>',
 3: '-',
 4: '0',
 5: '1',
 6: '2',
 7: '5',
 8: 'a',
 9: 'b',
 10: 'c',
 11: 'd',
 12: 'e',
 13: 'f',
 14: 'g',
 15: 'h',
 16: 'i',
 17: 'j',
 18: 'k',
 19: 'l',
 20: 'm',
 21: 'n',
 22: 'o',
 23: 'p',
 24: 'q',
 25: 'r',
 26: 's',
 27: 't',
 28: 'u',
 29: 'v',
 30: 'w',
 31: 'x',
 32: 'y',
 33: 'z'}

In [25]:
print(names[0])
tokens = ['<S>'] + list(names[0]) + ['<E>']
[reversed_voc[token] for token in tokens]

bulbasaur


[0, 9, 28, 19, 9, 8, 26, 8, 28, 25, 1]

In [26]:
from torch.nn.utils.rnn import pad_sequence
tok_names = [['<S>'] + list(name) + ['<E>'] for name in names]
max_length = max(map(len, tok_names))
encoded = [torch.tensor([reversed_voc[t] for t in tokens]) for tokens in tok_names]
batch = pad_sequence(encoded, batch_first=True, padding_value=reversed_voc['<P>'])

In [27]:
print(batch.shape)
print(batch[0])

torch.Size([1350, 29])
tensor([ 0,  9, 28, 19,  9,  8, 26,  8, 28, 25,  1,  2,  2,  2,  2,  2,  2,  2,
         2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2])


In [28]:
def positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    pos = torch.arange(0, max_len).unsqueeze(1)
    i = torch.arange(0, d_model, 2)
    denominator = 10000**(i/d_model)
    div = pos / denominator
    pe[:, 0::2] = torch.sin(div)
    pe[:, 1::2] = torch.cos(div)
    return pe
    

In [81]:
inputs = batch[:, :-1]
targets = batch[:, 1:]

In [82]:
print(inputs.shape, outputs.shape)

torch.Size([1350, 28]) torch.Size([1350, 28])


In [32]:
class TokenEmbedding(torch.nn.Module):
    def __init__(self, voc_size, d_model):
        super().__init__()
        self.d_model = d_model
        self.embedding = torch.nn.Embedding(voc_size, d_model)
        
    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)

In [34]:
class PositionalEncoding(torch.nn.Module):
    def __init__(self, d_model, max_len):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [84]:
class MultiHeadAttention(torch.nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.Q = torch.nn.Linear(d_model, d_model)
        self.K = torch.nn.Linear(d_model, d_model)
        self.V = torch.nn.Linear(d_model, d_model)

        self.out = torch.nn.Linear(d_model, d_model)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape

        Q = self.Q(x)
        K = self.K(x)
        V = self.V(x)

        Q = Q.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)

        scores = Q @ K.transpose(-2, -1)
        mask = torch.triu(
            torch.full((seq_len, seq_len), float('-inf')),
            diagonal=1
        )
        scores = scores + mask
        scores = torch.softmax(scores / math.sqrt(self.d_k), dim=-1)
        attn = scores @ V

        attn = attn.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)

        return self.out(attn)

In [85]:
class FeedForwardNetwork(torch.nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.layer1 = torch.nn.Linear(d_model, 4 * d_model)
        self.layer2 = torch.nn.Linear(4 * d_model, d_model)

    def forward(self, x):
        x = self.layer1(x)
        x = torch.nn.functional.relu(x)
        x = self.layer2(x)
        return x

In [99]:
class DecoderBlock(torch.nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.att = MultiHeadAttention(d_model, n_heads)
        self.ff = FeedForwardNetwork(d_model)
        self.norm1 = torch.nn.LayerNorm(d_model)
        self.norm2 = torch.nn.LayerNorm(d_model)
        self.dropout1 = torch.nn.Dropout(dropout)
        self.dropout2 = torch.nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.dropout1(self.att(x))
        x = self.norm1(x)
        x = x + self.dropout2(self.ff(x))
        x = self.norm2(x)
        return x

In [87]:
class Transformer(torch.nn.Module):
    def __init__(self, d_model, voc_size, max_len, n_heads, N):
        super().__init__()
        self.emb = TokenEmbedding(voc_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.decoder = torch.nn.ModuleList([
            DecoderBlock(d_model, n_heads) for _ in range(N)
        ])
        self.linear = torch.nn.Linear(d_model, voc_size)
    def forward(self, x):
        x = self.emb(x)
        x = self.pos_enc(x)
        for d_block in self.decoder:
            x = d_block(x)
        return self.linear(x)        

In [107]:
from sklearn.model_selection import train_test_split

train_inputs, val_inputs, train_targets, val_targets = train_test_split(
    inputs, targets, test_size=0.15
)
train_loader = DataLoader(TensorDataset(train_inputs, train_targets), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(val_inputs, val_targets), batch_size=32, shuffle=False)

# 8 têtes au lieu de 64 → 512/8 = 64 dims par tête (au lieu de 8)
model = Transformer(d_model=256, voc_size=len(voc), max_len=max_length, n_heads=8, N=3)

# LR plus bas + warmup via scheduler
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, betas=(0.9, 0.98), eps=1e-9)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda step: min((step + 1) ** -0.5, (step + 1) * (4000 ** -1.5)) * (512 ** 0.5)
)

loss_fn = torch.nn.CrossEntropyLoss()
best_val_loss = float('inf')
patience = 20
no_improve = 0

for epoch in range(n_epochs):
    # --- Training ---
    model.train()
    total_loss = 0
    for batch_input, batch_target in train_loader:
        output = model(batch_input).view(-1, len(voc))
        target = batch_target.view(-1)
        loss = loss_fn(output, target)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    # --- Validation ---
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch_input, batch_target in val_loader:
            output = model(batch_input).view(-1, len(voc))
            target = batch_target.view(-1)
            val_loss += loss_fn(output, target).item()

    # Calcul des moyennes AVANT de les utiliser (bug corrigé)
    train_avg = total_loss / len(train_loader)
    val_avg = val_loss / len(val_loader)
    print(f"Epoch {epoch+1}/{n_epochs} — Train: {train_avg:.4f} — Val: {val_avg:.4f}")

    if val_avg < best_val_loss:
        best_val_loss = val_avg
        torch.save(model.state_dict(), 'best_model.pt')
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"Early stopping à l'epoch {epoch+1}")
            break

model.load_state_dict(torch.load('best_model.pt'))

Epoch 1/200 — Train: 3.1996 — Val: 3.1501
Epoch 2/200 — Train: 3.1156 — Val: 3.0169
Epoch 3/200 — Train: 2.9495 — Val: 2.8110
Epoch 4/200 — Train: 2.7184 — Val: 2.5408
Epoch 5/200 — Train: 2.4378 — Val: 2.2356
Epoch 6/200 — Train: 2.1362 — Val: 1.9330
Epoch 7/200 — Train: 1.8614 — Val: 1.6848
Epoch 8/200 — Train: 1.6489 — Val: 1.5120
Epoch 9/200 — Train: 1.5024 — Val: 1.3996
Epoch 10/200 — Train: 1.4036 — Val: 1.3204
Epoch 11/200 — Train: 1.3294 — Val: 1.2618
Epoch 12/200 — Train: 1.2773 — Val: 1.2184
Epoch 13/200 — Train: 1.2371 — Val: 1.1854
Epoch 14/200 — Train: 1.2051 — Val: 1.1587
Epoch 15/200 — Train: 1.1778 — Val: 1.1364
Epoch 16/200 — Train: 1.1566 — Val: 1.1169
Epoch 17/200 — Train: 1.1376 — Val: 1.1000
Epoch 18/200 — Train: 1.1200 — Val: 1.0852
Epoch 19/200 — Train: 1.1050 — Val: 1.0716
Epoch 20/200 — Train: 1.0903 — Val: 1.0599
Epoch 21/200 — Train: 1.0783 — Val: 1.0492
Epoch 22/200 — Train: 1.0666 — Val: 1.0395
Epoch 23/200 — Train: 1.0558 — Val: 1.0303
Epoch 24/200 — Train

<All keys matched successfully>

In [108]:
def generate(model, max_len, vocab, idx_to_char, temperature=1.0):
    model.eval()
    input_seq = torch.tensor([[vocab['<S>']]])
    generated = []

    with torch.no_grad():
        for _ in range(max_len):
            output = model(input_seq)
            last_logits = output[0, -1, :]
            probs = torch.nn.functional.softmax(last_logits / temperature, dim=-1)
            next_idx = torch.multinomial(probs, 1).item()

            if next_idx == vocab['<E>']:
                break

            generated.append(idx_to_char[next_idx])
            next_token = torch.tensor([[next_idx]])
            input_seq = torch.cat([input_seq, next_token], dim=1)

    model.train()
    return ''.join(generated)

# Teste différentes températures
for temp in [0.5, 0.8, 1.0, 1.2]:
    print(f"\n--- Température {temp} ---")
    for _ in range(5):
        print(generate(model, max_len=15, vocab=reversed_voc, 
                       idx_to_char=voc, temperature=temp))


--- Température 0.5 ---
corombe
spoliwkoo-galar
minior-mete
pynash
deranin-garinal

--- Température 0.8 ---
drachakrle-mega
venistee
sandow-galar
koraidon-grry
minior-dinivalu

--- Température 1.0 ---
togembr-slial
tiryphu-tinisu
labliti
coshier-max
fealettr

--- Température 1.2 ---
zhora
gorpole
rampigrous
coxeyssle-meta
schof2snde
